In [1]:
%reset -f
import os
import numpy as np
import pandas as pd
from tqdm import tqdm


In [2]:
# 读取"E:\Slacking\2025\workdir\QSAR-QSIIR-MD\mode2_new\DILI_ligands"下的文件列表
DILI = os.listdir(r"E:\Slacking\2025\GraduationProject\MBMMLAB\DILI\DILI_ligands")
# 整理成一个DataFrame，列名为"ligand"
DILI = pd.DataFrame(DILI, columns=["ligand"])
# 去除文件名中的".pdbqt"后缀
DILI["ligand"] = DILI["ligand"].str.replace(".pdbqt", "", regex=False)
# 写入到文件"DILI_ligands.txt"，不带索引和列名
DILI.to_csv("DILI_ligands.txt", index=False, header=False, sep="\t")

In [3]:
# 读取DrugRep_pockets.xlsx
df = pd.read_excel('DrugRep_pockets.xlsx', sheet_name='Sheet1')

In [4]:
df.head()

,GeneSymbol,Pocket1,Volume1,Center1,Size1,Pocket2,Volume2,Center2,Size2,Pocket3,...,Center3,Size3,Pocket4,Volume4,Center4,Size4,Pocket5,Volume5,Center5,Size5
0,AASS,Chain A ( SER152 ILE199 ARG214 ARG210 ARG191 I...,1651,"1.4,4.0,3.7","21,18,18",Chain A ( ILE14 MET1 TYR12 GLU6 MET19 GLY9 PRO...,510,"-17.1,-3.1,-20.4","16,19,13",Chain A ( GLU319 ILE352 PRO318 TYR316 PHE355 G...,...,"-2.3,-13.3,-7.3","11,18,10",Chain A ( ILE21 VAL32 GLN35 ILE39 LEU31 VAL17 ...,187,"-19.3,5.2,-11.9","16,10,13",Chain A ( ALA235 PRO237 ARG240 ALA299 GLN298 G...,185,"7.0,9.9,-25.0","13,13,9"
1,ABCB1,Chain A ( VAL45 VAL46 PHE29 ASN36 ARG44 GLU35 ...,216,"-0.2,7.0,-2.4","17,12,13",Chain A ( ALA54 ALA55 PHE63 ASN36 ILE37 LEU28 ...,166,"1.3,-2.3,-5.8","14,11,15",Chain A ( LEU19 HSD18 LEU15 TRP14 ILE7 LEU2 AS...,...,"-4.8,0.2,20.7","11,13,15",Chain A ( ILE51 ALA55 ILE37 GLN48 ILE33 ILE64 ...,88,"8.5,1.9,-8.1","12,9,10",Chain A ( VAL46 ILE51 ASP41 ARG44 ILE37 ALA54 ...,82,"6.3,2.5,2.4","13,11,10"
2,ABCC1,Chain A ( LEU70 TYR15 ASN25 LYS23 ALA18 GLU59 ...,849,"-0.1,1.9,-4.6","22,12,22",Chain A ( THR81 MET1 THR77 THR75 MET8 ASN5 GLY...,335,"15.9,-0.0,-15.4","15,13,14",Chain A ( LEU34 PHE49 ILE37 ILE33 LEU40 VAL53 ...,...,"-10.0,-2.4,14.3","12,11,12",Chain A ( VAL53 ARG57 MET30 LEU54 ARG26 ),42,"-6.3,-3.2,8.4","9,10,8",Chain A ( PHE49 GLU32 LEU29 ILE33 LEU40 GLY36 ),21,"-14.4,4.3,13.3","9,8,12"
3,ABI1,Chain A ( ALA356 SER387 GLY323 ASP318 GLU334 T...,733,"-10.1,-20.2,-15.2","19,15,18",Chain A ( ALA18 LEU26 SER57 ASN25 VAL29 GLU21 ...,141,"-13.5,2.3,40.1","16,11,12",Chain A ( LEU68 ALA18 ALA67 GLY15 ASN71 TYR63 ...,...,"-6.0,3.1,30.3","11,10,11",Chain A ( TYR53 LEU26 SER57 THR50 VAL29 THR54 ...,51,"-22.9,0.1,41.0","10,9,9",Chain A ( LEU350 THR377 LYS344 ASN374 GLU349 S...,47,"-17.9,-17.8,-29.7","10,10,15"
4,ACOX3,Chain A ( PHE53 SER4 MET1 PRO9 LEU31 ILE52 PRO...,310,"-12.1,6.3,12.1","16,17,13",Chain A ( HSD45 ALA36 VAL37 PHE33 PHE58 ILE57 ...,123,"-7.9,2.0,-2.8","11,16,11",Chain A ( ASN104 PHE58 LEU22 LEU15 LYS25 HSD29...,...,"-7.9,-5.4,11.0","10,11,14",Chain A ( ILE49 PHE53 MET1 PHE2 TYR38 SER35 SE...,70,"-9.2,10.0,5.9","11,10,9",Chain A ( ILE49 PHE53 MET1 PHE2 LEU46 TYR38 GL...,67,"-15.5,8.9,3.8","9,12,12"


In [5]:
# 将DILI_ligands中的文件名列成列表
ligands = os.listdir('DILI_ligands')
# ligand为文件名，去掉后缀
ligands = [ligand.split('.')[0] for ligand in ligands]

In [6]:
len(ligands)

953

In [7]:
# 将GeneSymbol写入txt，兼容linux
with open('receptors.txt', 'w') as f:
    for gene in df['GeneSymbol']:
        f.write(gene + '\n')


In [8]:
# 将ligands写入txt
with open('ligands.txt', 'w') as f:
    for ligand in ligands:
        f.write(ligand + '\n')

In [10]:
# commands_DILI_pocket1.txt
# ./vina_1.2.7_linux_x86_64 --config ./pocket1/{GeneSymbol}.conf --receptor ./receptors/{GeneSymbol}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket1/{GeneSymbol}/{GeneSymbol}_{ligand}.txt
# 建立for循环，生成每个配体和每个受体的vina命令
commands = []
for gene in df['GeneSymbol']:
    for ligand in ligands:
        command = f'./vina_1.2.7_linux_x86_64 --config ./pocket1/{gene}.conf --receptor ./receptors/{gene}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket1/{gene}/{gene}_{ligand}.txt'
        commands.append(command)
# 写入txt
with open('commands_DILI_pocket1.txt', 'w') as f:
    for command in commands:
        f.write(command + '\n')

In [23]:
def generate_commands(df, ligands, pocket_number,sets):
    commands = []
    for gene in df['GeneSymbol']:
        for ligand in ligands:
            command = f'./vina_1.2.7_linux_x86_64 --config ./pocket{pocket_number}/{gene}.conf --receptor ./receptors/{gene}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket{pocket_number}/{gene}/{gene}_{ligand}.txt'
            commands.append(command)
    with open(f'commands_{sets}/commands_DILI_pocket{pocket_number}_{sets}.txt', 'w') as f:
        for command in commands:
            f.write(command + '\n')

In [ ]:
# ligands拆分成100个一份，直到最后一份，命名为ligands_1, ligands_2, ...
# 然后分别调用generate_commands函数，生成对应的命令文件
ligands_split = [ligands[i:i + 100] for i in range(0, len(ligands), 100)]
for i, ligands_subset in enumerate(ligands_split):
    set_number = i + 1
    # 新建commands_{set_number}文件夹
    os.makedirs(f'commands_{set_number}', exist_ok=True)
    generate_commands(df, ligands_subset, 1, set_number)
    generate_commands(df, ligands_subset, 2, set_number)
    generate_commands(df, ligands_subset, 3, set_number)
    generate_commands(df, ligands_subset, 4, set_number)
    generate_commands(df, ligands_subset, 5, set_number)

In [ ]:
# # commands_DILI_pocket2.txt
# # ./vina_1.2.7_linux_x86_64 --config ./pocket2/{GeneSymbol}.conf --receptor ./receptors/{GeneSymbol}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket2/{GeneSymbol}/{GeneSymbol}_{ligand}.txt
# # 建立for循环，生成每个配体和每个受体的vina命令
# commands = []
# for gene in df['GeneSymbol']:
#     for ligand in ligands:
#         command = f'./vina_1.2.7_linux_x86_64 --config ./pocket2/{gene}.conf --receptor ./receptors/{gene}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket2/{gene}/{gene}_{ligand}.txt'
#         commands.append(command)
# # 写入txt
# with open('commands_DILI_pocket2.txt', 'w') as f:
#     for command in commands:
#         f.write(command + '\n')
        
# # 整理成函数，输入为df和ligands列表


In [ ]:
# # commands_DILI_pocket3.txt
# # ./vina_1.2.7_linux_x86_64 --config ./pocket3/{GeneSymbol}.conf --receptor ./receptors/{GeneSymbol}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket3/{GeneSymbol}/{GeneSymbol}_{ligand}.txt
# # 建立for循环，生成每个配体和每个受体的vina命令
# commands = []
# for gene in df['GeneSymbol']:
#     for ligand in ligands:
#         command = f'./vina_1.2.7_linux_x86_64 --config ./pocket3/{gene}.conf --receptor ./receptors/{gene}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket3/{gene}/{gene}_{ligand}.txt'
#         commands.append(command)
# # 写入txt
# with open('commands_DILI_pocket3.txt', 'w') as f:
#     for command in commands:
#         f.write(command + '\n')

In [ ]:
# # commands_DILI_pocket4.txt
# # ./vina_1.2.7_linux_x86_64 --config ./pocket4/{GeneSymbol}.conf --receptor ./receptors/{GeneSymbol}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket4/{GeneSymbol}/{GeneSymbol}_{ligand}.txt
# # 建立for循环，生成每个配体和每个受体的vina命令
# commands = []
# for gene in df['GeneSymbol']:
#     for ligand in ligands:
#         command = f'./vina_1.2.7_linux_x86_64 --config ./pocket4/{gene}.conf --receptor ./receptors/{gene}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket4/{gene}/{gene}_{ligand}.txt'
#         commands.append(command)
# # 写入txt
# with open('commands_DILI_pocket4.txt', 'w') as f:
#     for command in commands:
#         f.write(command + '\n')

In [ ]:
# # commands_DILI_pocket5.txt
# # ./vina_1.2.7_linux_x86_64 --config ./pocket5/{GeneSymbol}.conf --receptor ./receptors/{GeneSymbol}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket5/{GeneSymbol}/{GeneSymbol}_{ligand}.txt
# # 建立for循环，生成每个配体和每个受体的vina命令
# commands = []
# for gene in df['GeneSymbol']:
#     for ligand in ligands:
#         command = f'./vina_1.2.7_linux_x86_64 --config ./pocket5/{gene}.conf --receptor ./receptors/{gene}_removeLigSol.pdbqt --ligand ./DILI_ligands/{ligand}.pdbqt > ./outputs/pocket5/{gene}/{gene}_{ligand}.txt'
#         commands.append(command)
# # 写入txt
# with open('commands_DILI_pocket5.txt', 'w') as f:
#     for command in commands:
#         f.write(command + '\n')